In [ ]:
import sys
import pandas as pd
import numpy as np
import cufflinks as cf
cf.go_offline()
sys.path.append('../../')

import bitcoin.storage.snapshots as ss
import bitcoin.storage.util as sutil

In [ ]:
yesterday = pd.datetime.now() - pd.offsets.Day(7)
spread = pd.read_sql('select * from {} where time > "{}"'.format(ss.SPREAD_TBL_NAME, yesterday), 
                     con=sutil.ENGINE, 
                     index_col='time')
print spread.shape
spread.tail()

In [ ]:
cols = ['gdax_bid', 'gdax_ask', 'bitstamp_bid', 'bitstamp_ask']
spread = spread[cols]
spread.iplot()

In [ ]:
(spread.gdax_ask - spread.gdax_bid).iplot()

In [ ]:
gdax = (spread.gdax_ask + spread.gdax_bid) / 2.
bitstamp = (spread.bitstamp_ask + spread.bitstamp_bid) / 2.
mid_price = pd.DataFrame(dict(gdax=gdax, bitstamp=bitstamp))
mid_price.iplot(kind='spread')

In [ ]:
#spread['2017-07-26 09:51'].iplot()

In [ ]:
cheap_gdax = spread.bitstamp_bid - (spread.gdax_bid + 0.01) - (0.0025 * spread.bitstamp_bid)
expensive_gdax = (spread.gdax_ask - 0.01) - spread.bitstamp_ask - (0.0025 * spread.bitstamp_ask)

(cheap_gdax).iplot(title='Bitstamp sell (market) - Gdax buy (limit) - fees (on bitstamp)', fill=True)
(expensive_gdax).iplot(title='Gdax sell (limit) - Bitstamp buy (market) - fees (on bitstamp)', fill=True)

In [ ]:
combined = pd.DataFrame({
    'cheap_gdax': cheap_gdax, 
    'expensive_gdax': expensive_gdax
})
combined.iplot(title='Cheap Gdax vs expensive Gdax (relative to bitstamp)', fill=True)

In [ ]:
bins = 10
prices = spread.gdax_bid

groups = prices.groupby(pd.TimeGrouper(freq='1T'))
past = groups.mean().diff()
fwd = past.shift(-1)
past_fwd = pd.DataFrame(dict(past=past, fwd=fwd))
past_fwd = past_fwd[~past_fwd.past.isnull()]

chunks = np.array_split(past_fwd.sort_values('past'), bins)
deciles = pd.DataFrame([chunk.mean() for chunk in chunks])
deciles.iplot(kind='bar', title='TS Mom Price Differences')
deciles